In [1]:
import sys
from pathlib import Path

for base in (Path.cwd(), Path.cwd().parent):
    src = base / "src"
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))


# LLM / RAG Evaluation
Cost-bounded judging with Driftalytics. This notebook uses a deterministic demo
provider so it runs offline. Real runs use `openai`, `anthropic`, or `ollama`.

In [2]:
from driftalytics.llm import LLMJudge


class DemoProvider:
    """Canned judge response for an offline notebook run."""

    def complete(self, prompt):
        return '{"faithfulness": 0.9, "answer_relevance": 0.85}', 42


questions = ["What is the capital of France?", "Who wrote Hamlet?"]
contexts = ["France's capital is Paris.", "Hamlet was written by Shakespeare."]
answers = ["Paris", "William Shakespeare"]

judge = LLMJudge(
    provider=DemoProvider(),
    metrics=["faithfulness", "answer_relevance"],
    budget_tokens=10_000,
    sample_rate=1.0,
    cache=True,
)
scores = judge.evaluate(questions, contexts, answers)
for question, score in zip(questions, scores):
    print(question, "->", score)

What is the capital of France? -> {'faithfulness': 0.9, 'answer_relevance': 0.85, '_status': 'ok', '_error': None}
Who wrote Hamlet? -> {'faithfulness': 0.9, 'answer_relevance': 0.85, '_status': 'ok', '_error': None}


In [3]:
second_pass = judge.evaluate(questions, contexts, answers)
print("statuses:", [item["_status"] for item in second_pass])
print("budget remaining:", judge.budget_remaining)

statuses: ['cached', 'cached']
budget remaining: 9916


In [4]:
import pandas as pd
from driftalytics import SlidingWindowRunner
from driftalytics.monitors.llm_monitors import FaithfulnessMonitor

reference = pd.DataFrame({"value": [0.0]})
batch = pd.DataFrame(
    {
        "question": questions,
        "context": contexts,
        "answer": answers,
    }
)
runner = SlidingWindowRunner(
    reference=reference,
    monitors=[FaithfulnessMonitor(judge=judge, threshold=0.5)],
)
print(runner.run(batch).summary())

Driftalytics report: 2 rows, 2 analyzed, 0 skipped, 0 alert(s), 1 window(s)
  faithfulness: score=0.1000 threshold=0.5000 alert=False status=ok
